In [1]:
%load_ext autoreload
%autoreload 2

In [101]:
import datatoolbox as dt
import pyam
import pandas as pd
import numpy as np
from data_shepherd import energy_forms, emissions
from pandas_indexing import isin, ismatch, concat

from pathlib import Path
import os

In [3]:

BOX_MOUNT_PATH = Path("~/Library/CloudStorage/Box-Box").expanduser()
if not BOX_MOUNT_PATH.is_dir():
    BOX_MOUNT_PATH = Path("~/Box").expanduser()

DOWNSCALING_DATA_PATH = (
    "Climate Policy Team/02 - Projects/IKEA NDC 1.5° Pathways 23-25 - phase II/"
    "2 - Work Packages/WP2 - National 1.5°C Pathways/"
    "Downscaling/DSCALE"
)

raw_data: os.PathLike = BOX_MOUNT_PATH / DOWNSCALING_DATA_PATH / "data" / "step0_raw_data"
input_data: os.PathLike = BOX_MOUNT_PATH / DOWNSCALING_DATA_PATH / "data" / "step1_input_data_for_DSCALE"
project_data: os.PathLike = BOX_MOUNT_PATH / DOWNSCALING_DATA_PATH / "data" / "step2_project_folder"

### Functions

In [4]:
import pycountry


# Manual overrides for known alternative names, common names, or mismatches

manual_overrides = {
    'england & wales': 'GBR',
    'scotland': 'GBR',
    'northern ireland': 'GBR',
    'reunion': 'REU',
    'timor leste': 'TLS',
    'congo dem rep': 'COD',
    'dutch caribbean': 'BES',  # Netherlands Caribbean
    'ascension island': 'ASC',
    'diego garcia': 'IOT',
    'tristan da cunha': 'SHN',
    'st vincent & grenadines': 'VCT',
    'st helena': 'SHN',
    'st lucia': 'LCA',
    'st pierre & miquelon': 'SPM',
    'line islands': 'KIR',
    'channel islands': 'JEY',  # or GGY for Guernsey
    'northern marianas': 'MNP',
    'south georgia': 'SGS',
    'wallis & futuna': 'WLF',
    'imn': 'IMN',  # Isle of Man
    'fsm': 'FSM',  # Federated States of Micronesia
    'swaziland': 'SZL',
    'cape verde': 'CPV',
    'saint-martin': 'MAF',
    'trinidad & tobago': 'TTO',
    'antigua & barbuda': 'ATG',
    'st kitts & nevis': 'KNA',
    'sao tome & principe': 'STP',
    'falkland islands': 'FLK',
    'turks & caicos': 'TCA',
    'ascension island': 'ASC',
    'line islands': 'KIR',
    'diego garcia': 'IOT',
    'st helena': 'SHN',
    'st lucia': 'LCA',
    'st pierre & miquelon': 'SPM',
    'channel islands': 'JEY',   # Jersey
    'northern marianas': 'MNP',
    'south georgia': 'SGS',
    'wallis & futuna': 'WLF',
    'american samoa': 'ASM',
    'reunion': 'REU',
    'timor leste': 'TLS',



    "bosniaherz":"BIH",
    "burkinafaso":"BFA",
    "brunei":"BRN",
    "czech":"CZE", 
    "turkey": "TUR",
    "congo_drc":"COD", 
    "coteivoire": "CIV",
    "cote d'ivoire":"CIV",
    "south korea": "KOR",
    "eu27":"EU27",
    "north korea": "PRK",

    "russia": "RUS",

    "viet nam": "VNM", # Pycountry uses 'Viet Nam'

    "czech republic": "CZE", # Sometimes confused with Czechia

    "usa": "USA",

    "uk": "GBR",

    "united kingdom": "GBR",

    "micronesia": "FSM",  # Federated States of Micronesia

    "united states virgin islands": "VIR",

    "democratic republic of the congo": "COD",

    "palestine": "PSE",

    "kosovo": "XKX",  # Kosovo is not officially recognized by ISO, but XKX is widely used

    "vatican":"VAT",

    "turkiye":"TUR", 
    "costarica":"CRI", 
    "hongkong":"HKG", 
    "korea":"KOR", 
    "southafrica":"ZAF", 
    "saudiarabia":"SAU", 
    "srilanka":"SRI", 
    "syria":"SYR",
    "newzealand":"NZL",

    "laos":"LAO", 
    "bosnia-herzegovina":"BIH", 
    "macedonia":"MKD",

    "timor leste":"TLS",
    'chinareg': 'CHN',
    'congo_repub': 'COG',
    'dominicanrep': 'DOM',
    'elsalvador': 'SLV',
    'eqguinea': 'GNQ',
    'iran': 'IRN',
    'koreadpr': 'PRK',
    'lao': 'LAO',
    'northmaced': 'MKD',
    'sri': 'LKA',
    'ssudan': 'SSD',
    'taipei': 'TWN',
    'trinidad': 'TTO',
    'uae': 'ARE',
    'curacao':'CUW',

}

 

def get_iso_code(country_name):

    country_name_lower = country_name.strip().lower()

 

    # First, check manual overrides

    if country_name_lower in manual_overrides:

        return manual_overrides[country_name_lower]

 

    # Then check pycountry data

    for country in pycountry.countries:

        names_to_check = [

            country.name.lower(),

            getattr(country, 'official_name', '').lower(),

            getattr(country, 'common_name', '').lower() if hasattr(country, 'common_name') else ''

        ]

        if country_name_lower in names_to_check:

            return country.alpha_3

 

    # If still not found, return original input or None

    return country_name

 


# Example usage

country_name = "Turkey" # This one creates problems without `manual_overrides `

iso_code = get_iso_code(country_name)

print(f"The ISO code for '{country_name}' is {iso_code}.")


The ISO code for 'Turkey' is TUR.


# HISTORICAL DATA

## PRIMAP 2025

In [122]:
primap_2024 = pd.read_csv(raw_data / "Guetschow_et_al_2024-PRIMAP-hist_v2.5.1_final_no_extrap_no_rounding_27-Feb-2024.csv")
primap_2025 = pd.read_csv(raw_data / "Guetschow_et_al_2025a-PRIMAP-hist_v2.7_final_no_extrap_no_rounding_22-Aug-2025.csv")

In [135]:
primap_2025['scenario (PRIMAP-hist)'].unique()

array(['HISTCR', 'HISTTP'], dtype=object)

In [136]:
primap_2024['scenario (PRIMAP-hist)'].unique()

array(['HISTCR', 'HISTTP'], dtype=object)

In [137]:
primap_2025=primap_2025.replace({
    'CH4 * gigagram / yr':'CH4 * gigagram / a', 
    'CO2 * gigagram / yr':'CO2 * gigagram / a',
    'N2O * gigagram / yr':'N2O * gigagram / a', 
    'NF3 * gigagram / yr':'NF3 * gigagram / a',
    'SF6 * gigagram / yr':'SF6 * gigagram / a'
})

In [140]:
primap_2025

,source,scenario (PRIMAP-hist),provenance,area (ISO3),entity,unit,category (IPCC2006_PRIMAP),1750,1751,1752,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,PRIMAP-hist_v2.7_final_ne_nr,HISTCR,derived,ABW,CH4,CH4 * gigagram / a,0,0.014002,0.014107,0.014212,...,0.738639,0.747826,0.751525,0.757178,0.764758,0.762113,0.774698,0.777828,0.784228,NaN
1,PRIMAP-hist_v2.7_final_ne_nr,HISTCR,derived,ABW,CH4,CH4 * gigagram / a,1,0.005636,0.005672,0.005708,...,0.119856,0.125926,0.125206,0.126509,0.130455,0.124153,0.132606,0.132060,0.134182,NaN
2,PRIMAP-hist_v2.7_final_ne_nr,HISTCR,derived,ABW,CH4,CH4 * gigagram / a,1.A,0.005636,0.005672,0.005708,...,0.054473,0.058383,0.056130,0.056356,0.060196,0.055070,0.058487,0.054671,0.056793,NaN
3,PRIMAP-hist_v2.7_final_ne_nr,HISTCR,derived,ABW,CH4,CH4 * gigagram / a,1.B,0.000000,0.000000,0.000000,...,0.065383,0.067543,0.069075,0.070153,0.070259,0.069083,0.074119,0.077389,0.077389,NaN
4,PRIMAP-hist_v2.7_final_ne_nr,HISTCR,derived,ABW,CH4,CH4 * gigagram / a,1.B.1,0.000000,0.000000,0.000000,...,0.065383,0.067543,0.069075,0.070153,0.070259,0.069083,0.074119,0.077389,0.077389,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55683,PRIMAP-hist_v2.7_final_ne_nr,HISTTP,derived,ZWE,PFCS (SARGWP100),CO2 * gigagram / a,2,0.000000,0.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
55684,PRIMAP-hist_v2.7_final_ne_nr,HISTTP,derived,ZWE,PFCS (SARGWP100),CO2 * gigagram / a,M.0.EL,0.000000,0.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
55685,PRIMAP-hist_v2.7_final_ne_nr,HISTTP,derived,ZWE,SF6,SF6 * gigagram / a,0,0.000000,0.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
55686,PRIMAP-hist_v2.7_final_ne_nr,HISTTP,derived,ZWE,SF6,SF6 * gigagram / a,2,0.000000,0.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [141]:
primap_2025.to_csv(input_data / "Guetschow_et_al_2025a-PRIMAP-hist_v2.7_final_no_extrap_no_rounding_22-Aug-2025.csv", index=False)

## input_reference_iea_2022: IEA WEB 2025 from data_shepherd

In [72]:
ds_iea=energy_forms.get_historic_iea_web(source = "IEA_WEB_DETAILED_2024")

pyam.core - INFO: `['Primary Energy', 'Primary Energy|Biomass', 'Primary Energy|Coal', 'Primary Energy|Gas', 'Primary Energy|Oil']` - 4598 of 24375 rows are not aggregates of components
data_shepherd.energy_forms - WARNING:  - Historic: Primary Energy has wrong aggregations in Primary Energy, reaggregating from the leafs:
                                      variable  components        abs       rel  total*0.01
region variable       unit    year                                                         
AGO    Primary Energy EJ / yr 1990    0.246356    0.248964  -0.002608 -0.010586    0.002490
                              1991    0.252255    0.255033  -0.002778 -0.011013    0.002550
                              1992    0.253662    0.256686  -0.003024 -0.011921    0.002567
                              1993    0.262708    0.265912  -0.003204 -0.012196    0.002659
                              1994    0.266300    0.269521  -0.003221 -0.012095    0.002695
                              19

In [73]:
ds_iea.convert_unit("TWh / yr", "EJ/yr", inplace=True)

In [74]:
ds_iea.aggregate("Primary Energy|Fossil", [
    'Primary Energy|Coal', 
    'Primary Energy|Gas', 
    'Primary Energy|Oil'
], append=True)

In [75]:
ds_iea.aggregate("Secondary Energy|Electricity|Fossil", [
    'Secondary Energy|Electricity|Coal', 
    'Secondary Energy|Electricity|Gas', 
    'Secondary Energy|Electricity|Oil'
], append=True)

In [103]:
ds_iea.filter(region = "ZAF", variable = ["Final Energy", "Final Energy|Electricity"], year = [2010,2015]).timeseries()

2010  \
model                 scenario region variable                 unit                
IEA_WEB_DETAILED_2024 Historic ZAF    Final Energy             EJ / yr  4.088008   
                                      Final Energy|Electricity EJ / yr  0.729659   

                                                                            2015  
model                 scenario region variable                 unit               
IEA_WEB_DETAILED_2024 Historic ZAF    Final Energy             EJ / yr  4.432918  
                                      Final Energy|Electricity EJ / yr  0.688781

In [102]:
ds_iea.filter(region = "ZAF", variable = ["Secondary Energy|Electricity|Coal","Secondary Energy|Electricity|Oil", "Secondary Energy|Electricity"], year = [2010,2015]).timeseries()

2010  \
model                 scenario region variable                          unit              
IEA_WEB_DETAILED_2024 Historic ZAF    Secondary Energy|Electricity      EJ/yr  0.931666   
                                      Secondary Energy|Electricity|Coal EJ/yr  0.871204   
                                      Secondary Energy|Electricity|Oil  EJ/yr  0.000709   

                                                                                   2015  
model                 scenario region variable                          unit             
IEA_WEB_DETAILED_2024 Historic ZAF    Secondary Energy|Electricity      EJ/yr  0.907114  
                                      Secondary Energy|Electricity|Coal EJ/yr  0.806483  
                                      Secondary Energy|Electricity|Oil  EJ/yr  0.017093

In [87]:
df_iea = ds_iea.timeseries()

In [88]:
df_iea=df_iea.rename(index = {"IEA_WEB_DETAILED_2024":"IEA", "Historic":"Historic data"})

In [89]:
df_iea.index.names =  ['MODEL', 'SCENARIO', 'REGION', 'VARIABLE', 'UNIT']

In [90]:
df_iea=df_iea.reset_index()

In [91]:
df_iea=df_iea.replace({"EJ / yr":"EJ/yr"})

In [92]:
# EST has negative Oil values, we abs them
df_iea_est = df_iea.loc[(df_iea["REGION"]=="EST") & (df_iea["VARIABLE"].isin(["Primary Energy|Oil|w/o CCS", "Primary Energy|Oil"]))]
df_iea = pd.concat([
    df_iea.drop([
        8098, #Primary Oil|w/o CCS
        8096]), #Primary Oil
    abs(df_iea_est.set_index(['MODEL', 'SCENARIO',   'REGION', 'VARIABLE',     'UNIT'])).reset_index()
])

In [93]:
df_iea.to_csv(input_data / "input_reference_iea_2022.csv",index=False)

## Extended_IEA_en_bal_2019_ISO - from IEA WEB_2025_BIG

In [128]:
# TODO: filter out what is not used - you can help yourself with fixture.py 

In [112]:
flow_mapp_dict={
  "INDPROD": "Production", 
  "IMPORTS": "Imports",
  "EXPORTS": "Exports", 
  "MARBUNK": "International marine bunkers", 
  "AVBUNK": "International aviation bunkers", 
  "STOCKCHA": "Stock changes", 
  # "TES": "Total primary energy supply", 
  "TRANSFER": "Transfers", 
  "STATDIFF": "Statistical differences", 
  "TOTTRANF": "Transformation processes", 
  "MAINELEC": "Main activity producer electricity plants", 
  "AUTOELEC": "Autoproducer electricity plants", 
  "MAINCHP": "Main activity producer CHP plants", 
  "AUTOCHP": "Autoproducer CHP plants", "MAINHEAT": 
  "Main activity producer heat plants", "AUTOHEAT": 
  "Autoproducer heat plants", "EPUMPST": "Heat pumps", 
  "EPOWERPLT": "Electric boilers", 
  "TGTL": "Chemical heat for electricity production", 
  "TBLASTFUR": "Blast furnaces", 
  "TGASWKS": "Gas works", 
  "TCOKEOVS": "Coke ovens", 
  "TPATFUEL": "Patent fuel plants", 
  "TBKB": "BKB/peat briquette plants", 
  "TREFINER": "Oil refineries", 
  "TPETCHEM": "Petrochemical plants", 
  "TCOALLIQ": "Coal liquefaction plants",
  "EGTL": "Gas-to-liquids (GTL) plants", 
  "TBLENDGAS": "For blended natural gas", 
  "TCHARCOAL": "Charcoal production plants", 
  "TNONSPEC": "Non-specified (transformation)", 
  "TOTENGY": "Energy industry own use", 
  "EMINES": "Coal mines", 
  "EOILGASEX": "Oil and gas extraction", 
  "EBIOGAS": "Gasification plants for biogases", 
  "ELNG": "Liquefaction (LNG) / regasification plants", 
  "EPUMPST": " Own use in electricity, CHP and heat plants", 
  "EPUMPST": "Pumped storage plants", 
  "ENUC": "Nuclear industry", 
  "ENONSPEC": "Non-specified (energy)", 
  "DISTLOSS": "Losses", "TFC": 
  "Total final consumption", 
  "TOTIND": "Industry", 
  "MINING": "Mining and quarrying", 
  "CONSTRUC": "Construction", 
  "MANUFACT": "Manufacturing", 
  "IRONSTL": "Iron and steel", 
  "CHEMICAL": "Chemical and petrochemical", 
  "NONFERR": "Non-ferrous metals", 
  "NONMET": "Non-metallic minerals", 
  "TRANSEQ": "Transport equipment", 
  "MACHINE": "Machinery", 
  "FOODPRO": "Food and tobacco", 
  "PAPERPRO": " Paper, pulp and printing", 
  "WOODPRO": "Wood and wood products", 
  "TEXTILES": "Textile and leather", 
  "INONSPEC": "Industry not elsewhere specified", 
  "TOTTRANS": "Transport", 
  "WORLDAV": "World aviation bunkers", 
  "DOMESAIR": "Domestic aviation", 
  "ROAD": "Road", 
  "RAIL": "Rail", 
  "PIPELINE": "Pipeline transport", 
  "WORLDMAR": "World marine bunkers", 
  "DOMESNAV": "Domestic navigation", 
  "TRNONSPE": "Non-specified (transport)", 
  "RESIDENT": "Residential", 
  "COMMPUB": "Commercial and public services", 
  "AGRICULT": "Agriculture/forestry", 
  "FISHING": "Fishing", 
  "ONONSPEC": "Final consumption not elsewhere specified", 
  "NONENUSE": "Non-energy use", 
  "NEINTREN": "Non-energy use industry/transformation/energy", 
  "NEIND": "Memo: Non-energy use in industry", 
  "NECONSTRUC": "Memo: Non-energy use in construction", 
  "NEMINING": "Memo: Non-energy use in mining and quarrying", ""
  "NEIRONSTL": "Memo: Non-energy use in iron and steel", 
  "NECHEM": "Memo: Non-energy use in chemical/petrochemical", 
  "NENONFERR": "Memo: Non-energy use in non-ferrous metals", 
  "NENONMET": "Memo: Non-energy use in non-metallic minerals", 
  "NETRANSEQ": "Memo: Non-energy use in transport equipment", 
  "NEMACHINE": "Memo: Non-energy use in machinery", 
  "NEFOODPRO": "Memo: Non-energy use in food/beverages/tobacco", 
  "NEPAPERPRO": "Memo: Non-energy use in paper/pulp and printing", 
  "NEWOODPRO": "Memo: Non-energy use in wood and wood products", 
  "NETEXTILES": "Memo: Non-energy use in textiles and leather", 
  "NEINONSPEC": "Memo: Non-energy use in industry not elsewhere specified", 
  "NETRANS": "Non-energy use in transport", 
  "NEOTHER": "Non-energy use in other", 
  "ELOUTPUT": "Electricity output (GWh)", 
  "ELMAINE": "Electricity output (GWh)-main activity producer electricity plants", 
  "ELAUTOE": "Electricity output (GWh)-autoproducer electricity plants", 
  "ELMAINC": "Electricity output (GWh)-main activity producer CHP plants", 
  "ELAUTOC": "Electricity output (GWh)-autoproducer CHP plants", 
  "HEATOUT": "Heat output", 
  "HEMAINC": "Heat output-main activity producer CHP plants", 
  "HEAUTOC": "Heat output-autoproducer CHP plants", 
  "HEMAINH": "Heat output-main activity producer heat plants", 
  "HEAUTOH": "Heat output-autoproducer heat plants" }

In [113]:
iea_flow_mapping_complement = {

    # Supply
    "TPES": "Total primary energy supply",
    "PRODUCTION": "Production",
    "IMPORTS": "Imports",
    "EXPORTS": "Exports",
    "TRANSFERS": "Transfers",
    "STATDIFF": "Statistical differences",
    "STCHANAT": "Stock changes",

    # Energy industry & extraction
    "COALMINES": "Coal mines",
    "OILGASEXTR": "Oil and gas extraction",
    "NUCLEAR": "Nuclear industry",
    "EREFINER": "Oil refineries",
    "GASWKS": "Gas works",
    "EBLASTFUR": "Blast furnaces",
    "ECOKEOVS": "Coke ovens",
    "EPATFUEL": "Patent fuel plants",
    "EBKB": "BKB/peat briquette plants",
    "ECHARCOAL": "Charcoal production plants",
    "ECOALLIQ": "Coal liquefaction plants",
    "PETCHEM": "Petrochemical plants",
    "GTL": "Gas-to-liquids (GTL) plants",
    "LNG": "Liquefaction (LNG) / regasification plants",
    "BIOGASIF": "Gasification plants for biogases",

    # Power & heat
    "ELECTRBOIL": "Electric boilers",
    "PUMPST": "Pumped storage plants",
    "THEAT": "Heat output",
    "ELOUTPUT": "Electricity output (GWh)",
    "AUTOELECOUT": "Electricity output (GWh)-autoproducer electricity plants",
    "AUTOCHPOUT": "Electricity output (GWh)-autoproducer CHP plants",
    "MAINELECOUT": "Electricity output (GWh)-main activity producer electricity plants",
    "MAINCHPOUT": "Electricity output (GWh)-main activity producer CHP plants",
    "AUTOCHPHEAT": "Heat output-autoproducer CHP plants",
    "AUTOHEAT": "Heat output-autoproducer heat plants",
    "MAINCHPHEAT": "Heat output-main activity producer CHP plants",
    "MAINHEAT": "Heat output-main activity producer heat plants",

    # Transformation & losses
    "TRANSF": "Transformation processes",
    "NONSPTRANS": "Non-specified (transformation)",
    "LOSS": "Losses",
    "TBOILER": "Transformation boilers",
    "CHEMHEAT": "Chemical heat for electricity production",

    # Energy industry own use
    "EINDUSE": "Energy industry own use",

    # Final consumption aggregates
    "IND": "Industry",
    "TRANS": "Transport",
    "NONEN": "Non-specified (energy)",
    "ONONSPEC": "Final consumption not elsewhere specified",

    # Transport – memo & world bunkers
    "WORLDAIR": "World aviation bunkers",
    "WORLDMAR": "World marine bunkers",
    "NONSPTRAN": "Non-specified (transport)",

    # Network energy (NE*) – IEA auxiliary balances
    "NE_TOT": "Network energy total",
    "NE_IND": "Network energy – industry",
    "NE_TRANS": "Network energy – transport",
    "NE_OTHER": "Network energy – other",
    "NE_IND_TRANSF": "Network energy – industry transformation",
    "NE_MINING": "Network energy – mining and quarrying",
    "NE_CHEM": "Network energy – chemical and petrochemical",
    "NE_IRONSTL": "Network energy – iron and steel",
    "NE_NONFERR": "Network energy – non-ferrous metals",
    "NE_NONMET": "Network energy – non-metallic minerals",
    "NE_MACHINE": "Network energy – machinery",
    "NE_TRANSEQ": "Network energy – transport equipment",
    "NE_FOODPRO": "Network energy – food and tobacco",
    "NE_PAPERPRO": "Network energy – paper, pulp and printing",
    "NE_TEXTILES": "Network energy – textile and leather",
    "NE_WOODPRO": "Network energy – wood and wood products",
    "NE_CONSTRUC": "Network energy – construction",
    "NE_INONSPEC": "Network energy – industry not elsewhere specified",

    # Miscellaneous
    "TELE": "Transmission and distribution losses",
    "FORBLENDNG": "For blended natural gas"
}


In [114]:
iea_product_mapping = {

    # Aggregates
    "TOTAL": "Total",
    "RENEWABLES_TOTAL": "Memo: Renewables",

    # Coal and coal products
    "HARDCOAL_ND": "Hard coal (if no detail)",
    "ANTHRACITE": "Anthracite",
    "SUB_BITCOAL": "Sub-bituminous coal",
    "OTH_BITCOAL": "Other bituminous coal",
    "BROWNCOAL_ND": "Brown coal (if no detail)",
    "COKING_COAL": "Coking coal",
    "LIGNITE": "Lignite",
    "PATENT_FUEL": "Patent fuel",
    "BKB": "BKB",
    "COKE_OVEN_COKE_OTH": "Coke oven coke",
    "GAS_COKE": "Gas coke",
    "COAL_TAR": "Coal tar",

    # Manufactured & recovered gases
    "GASWORKS_GAS": "Gas works gas",
    "COKE_OVEN_GAS": "Coke oven gas",
    "BLAST_FURNACE_GAS": "Blast furnace gas",
    "OTH_RECOVGASES": "Other recovered gases",
    "MANUFACTURED_GAS_OUTPUT": "Elec/heat output from non-specified manufactured gases",

    # Peat and shale
    "PEAT": "Peat",
    "PEAT_PRODUCTS": "Peat products",
    "OIL_SHALE": "Oil shale",
    "OIL_SHALE_SANDS": "Oil shale and oil sands",

    # Natural gas and derivatives
    "NATURAL_GAS": "Natural gas",
    "NGL": "Natural gas liquids",
    "ETHANE": "Ethane",

    # Crude oil and refinery inputs
    "OIL_PRIM_PRODUCTS_ND": "Crude/NGL/feedstocks (if no detail)",
    "CRUDE_OIL": "Crude oil",
    "REFINERY_FEEDSTOCKS": "Refinery feedstocks",

    # Other primary oil inputs
    "ADDITIVES": "Additives/blending components",
    "HYDROCARBONS_OTHER": "Other hydrocarbons",

    # Refinery outputs – gases
    "REFINERY_GAS": "Refinery gas",
    "LPG": "Liquefied petroleum gases (LPG)",

    # Transport fuels
    "MOTOR_GASOLINE_NONBIO": "Motor gasoline excl. biofuels",
    "AVIATION_GASOLINE": "Aviation gasoline",
    "GASOLINE_JET": "Gasoline type jet fuel",
    "KEROSENE_JET_NONBIO": "Kerosene type jet fuel excl. biofuels",
    "KEROSENE_JET_BIO": "Bio jet kerosene",
    "KEROSENE_OTHER": "Other kerosene",
    "GAS_DIESEL_OIL_NONBIO": "Gas/diesel oil excl. biofuels",

    # Other oil products
    "FUEL_OIL_RESIDUAL": "Fuel oil",
    "NAPHTHA": "Naphtha",
    "WHITE_SPIRIT": "White spirit & SBP",
    "LUBRICANTS": "Lubricants",
    "BITUMEN": "Bitumen",
    "PARAFFIN_WAXES": "Paraffin waxes",
    "PETROLEUM_COKE": "Petroleum coke",
    "OTH_SEC_OIL_PRODS_ND": "Non-specified oil products",

    # Waste (renewable and non-renewable)
    "WASTE_INDUSTRIAL_NONREN": "Industrial waste",
    "WASTE_MUNICIPAL_NONREN": "Municipal waste (non-renewable)",
    "WASTE_MUNICIPAL_REN": "Municipal waste (renewable)",

    # Bioenergy (memo items)
    "PRIMARY_SOLID_BIOFUEL": "Primary solid biofuels",
    "BIOGASES": "Biogases",
    "BIOGASOLINE": "Biogasoline",
    "BIODIESEL": "Biodiesels",
    "LIQBIOFUEL_OTHER": "Other liquid biofuels",
    "BIOFUEL_NONSPEC": "Non-specified primary biofuels and waste",
    "CHARCOAL": "Charcoal",

    # Electricity & heat
    "ELECTRICITY": "Electricity",
    "HEAT": "Heat",
    "HEAT_COMBUSTIBLES_NS": "Heat output from non-specified combustible fuels",

    # Renewables
    "HYDRO": "Hydro",
    "GEOTHERMAL": "Geothermal",
    "SOLAR_PV": "Solar photovoltaics",
    "SOLAR_THERMAL": "Solar thermal",
    "WIND": "Wind",
    "TIDAL_WAVE_OCEAN": "Tide, wave and ocean",

    # Other
    "NUCLEAR": "Nuclear",
    "OTH_ENSOURC": "Other sources"
}


In [115]:
ext_iea_web = pd.read_csv(
    raw_data / 'WEB_2025_BIG.csv', # comes from MCG ca_data_management IEA_WEB_DETAILED_2025
    encoding="latin1")

In [116]:
ext_iea_web=ext_iea_web.drop(0).rename(columns={"Unnamed: 0":"COUNTRY", "Unnamed: 1":"PRODUCT", "TIME":"FLOW"})

In [117]:
ext_iea_web["COUNTRY"]=ext_iea_web["COUNTRY"].str.capitalize()
# ext_iea_web["PRODUCT"]=ext_iea_web["PRODUCT"].str.capitalize()

In [118]:
ext_iea_web=ext_iea_web.replace(flow_mapp_dict)
ext_iea_web=ext_iea_web.replace(iea_flow_mapping_complement)
ext_iea_web=ext_iea_web.replace(iea_product_mapping)

In [119]:
mapping_dict = {c: get_iso_code(c) for c in ext_iea_web["COUNTRY"].unique()}
ext_iea_web["ISO"] = ext_iea_web["COUNTRY"].map(mapping_dict)

In [120]:
ext_iea_web=ext_iea_web.loc[ext_iea_web.ISO.str.len() == 3]

In [121]:
ext_iea_web=ext_iea_web.set_index(["ISO", "COUNTRY", "PRODUCT", "FLOW", "UNIT"])#.astype(float)

In [122]:
ext_iea_web=ext_iea_web.replace({"..":np.nan, "x":np.nan, "c":np.nan}).astype(float)

In [123]:
ext_iea_web= concat([
    ext_iea_web.loc[isin(UNIT = "GWH")],
    ext_iea_web.loc[~isin(UNIT = "GWH")] / 41.868
])

In [124]:
# ext_iea_web=ext_iea_web.loc[isin(ISO = iea_countries)]

In [125]:
ext_iea_web=ext_iea_web.droplevel("UNIT")

In [126]:
ext_iea_web=ext_iea_web.loc[~isin(ISO = "CHN", COUNTRY = "Chinareg")]

In [127]:
ext_iea_web.to_csv(input_data / "Extended_IEA_en_bal_2019_ISO.csv")

In [269]:
iea_countries = ['ALB', 'DZA', 'AGO', 'ARG', 'ARM', 'AUS', 'AUT', 'AZE', 'BHR', 'BGD', 'BLR', 'BEL', 'BEN', 'BOL', 'BIH', 'BWA', 'BRA', 'BRN', 'BGR', 'KHM', 'CMR', 'CAN', 'CHL', 'CHN', 'COL', 'COG', 'CRI', 'CIV', 'HRV', 'CUB', 'ANT', 'CYP', 'CZE', 'PRK', 'COD', 'DNK', 'DOM', 'ECU', 'EGY', 'SLV', 'ERI', 'EST', 'ETH', 'FIN', 'FRA', 'GAB', 'GEO', 'DEU', 'GHA', 'GIB', 'GRC', 'GTM', 'HTI', 'HND', 'HKG', 'HUN', 'ISL', 'IND', 'IDN', 'IRN', 'IRQ', 'IRL', 'ISR', 'ITA', 'JAM', 'JPN', 'JOR', 'KAZ', 'KEN', 'KOR', 'KWT', 'KGZ', 'LVA', 'LBN', 'LBY', 'LTU', 'LUX', 'MYS', 'MLT', 'TWN', 'MUS', 'MEX', 'MDA', 'MNG', 'MNE', 'MAR', 'MOZ', 'MMR', 'NAM', 
'NPL', 'NLD', 'NZL', 'NIC', 'NER', 'NGA', 'MKD', 'NOR', 'OMN', 'PAK', 'PAN', 'PRY', 'PER', 'PHL', 'POL', 'PRT', 'QAT', 'ROU', 'RUS', 'SAU', 'SEN', 'SRB', 'SGP', 'SVK', 'SVN', 'ZAF', 'SSD', 'ESP', 'LKA', 'SDN', 'SUR', 'SWE', 'CHE', 'SYR', 'TJK', 'TZA', 'THA', 'TGO', 'TTO', 'TUN', 'TUR', 'TKM', 'UKR', 'ARE', 'GBR', 'USA', 'URY', 'UZB', 'VEN', 'VNM', 'YEM', 'ZMB', 'ZWE', 'MLI', 'UGA']

In [271]:
# from downscaler.fixtures import iea_countries
print('Missing:', [x for x in iea_countries 
if x not in set(ext_iea_web.ISO.unique()) # your list
])

Missing: ['BIH', 'BRN', 'COG', 'CRI', 'CIV', 'ANT', 'CZE', 'PRK', 'COD', 'DOM', 'SLV', 'HKG', 'IRN', 'TWN', 'MDA', 'NZL', 'MKD', 'RUS', 'SAU', 'ZAF', 'SSD', 'LKA', 'SYR', 'TZA', 'TTO', 'TUR', 'ARE', 'GBR', 'USA', 'VNM']


## IEA_hist_trade_variables_v2022: Trade from IEA_WEB_DETAILED_2025 -> in datashelf

In [19]:
iea_trade = dt.findp(source = 'IEA_WEB_DETAILED_2025', variable = ["Energy exports**", "Energy imports**"]).as_pyam()

In [20]:
# BIOMASS TRADE
iea_trade.aggregate(
    'Energy exports|Primary Energy|Biomass|Volume', 
    ['Energy exports|Bio jet kerosene',
    'Energy exports|Biodiesels',
    'Energy exports|Biogases',
    'Energy exports|Biogasoline',
    'Energy exports|Non-specified primary biofuels and waste',
    'Energy exports|Other liquid biofuels',
    'Energy exports|Primary solid biofuels',
], append=True
) 

iea_trade.aggregate(
    'Energy imports|Primary Energy|Biomass|Volume', 
    [
        'Energy imports|Bio jet kerosene',
        'Energy imports|Biodiesels',
        'Energy imports|Biogases',
        'Energy imports|Biogasoline',
        'Energy imports|Non-specified primary biofuels and waste',
        'Energy imports|Other liquid biofuels',
        'Energy imports|Primary solid biofuels'
], append=True
) 

iea_trade.add("Energy exports|Primary Energy|Biomass|Volume", "Energy imports|Primary Energy|Biomass|Volume", "Trade|Primary Energy|Biomass|Volume", append=True)

In [21]:
# GAS TRADE
iea_trade.add("Energy exports|Natural gas", "Energy imports|Natural gas", "Trade|Primary Energy|Gas|Volume", append=True)

In [22]:
# COAL TRADE
iea_trade.aggregate(
    'Energy exports|Primary Energy|Coal|Volume', 
    [
    "Energy exports|Hard coal (if no detail)",
    "Energy exports|Brown coal (if no detail)",
    "Energy exports|Anthracite",
    "Energy exports|Coking coal",
    "Energy exports|Other bituminous coal",
    "Energy exports|Sub-bituminous coal",
    "Energy exports|Lignite",
    "Energy exports|Patent fuel",
    "Energy exports|Coke oven coke",
    "Energy exports|Gas coke",
    "Energy exports|Coal tar",
    "Energy exports|BKB",
    "Energy exports|Gas works gas",
    "Energy exports|Coke oven gas",
    "Energy exports|Blast furnace gas",
    "Energy exports|Other recovered gases",
    "Energy exports|Peat",
    "Energy exports|Peat products",
    "Energy exports|Oil shale and oil sands",

], append=True
) 

iea_trade.aggregate(
    'Energy imports|Primary Energy|Coal|Volume', 
    [
    "Energy imports|Hard coal (if no detail)",
    "Energy imports|Brown coal (if no detail)",
    "Energy imports|Anthracite",
    "Energy imports|Coking coal",
    "Energy imports|Other bituminous coal",
    "Energy imports|Sub-bituminous coal",
    "Energy imports|Lignite",
    "Energy imports|Patent fuel",
    "Energy imports|Coke oven coke",
    "Energy imports|Gas coke",
    "Energy imports|Coal tar",
    "Energy imports|BKB",
    "Energy imports|Gas works gas",
    "Energy imports|Coke oven gas",
    "Energy imports|Blast furnace gas",
    "Energy imports|Other recovered gases",
    "Energy imports|Peat",
    "Energy imports|Peat products",
    "Energy imports|Oil shale and oil sands",

], append=True
) 

iea_trade.add("Energy exports|Primary Energy|Coal|Volume", "Energy imports|Primary Energy|Coal|Volume", "Trade|Primary Energy|Coal|Volume", append=True)

In [23]:
# OIL TRADE
iea_trade.aggregate(
    'Energy exports|Primary Energy|Oil|Volume', 
    [
    "Energy exports|Crude/NGL/feedstocks (if no detail)",
    "Energy exports|Crude oil",
    "Energy exports|Natural gas liquids",
    "Energy exports|Refinery feedstocks",
    "Energy exports|Additives/blending components",
    "Energy exports|Other hydrocarbons",
    "Energy exports|Refinery gas",
    "Energy exports|Ethane",
    "Energy exports|Liquefied petroleum gases (LPG)",
    "Energy exports|Motor gasoline excl. biofuels",
    "Energy exports|Aviation gasoline",
    "Energy exports|Gasoline type jet fuel",
    "Energy exports|Kerosene type jet fuel excl. biofuels",
    "Energy exports|Other kerosene",
    "Energy exports|Gas/diesel oil excl. biofuels",
    "Energy exports|Fuel oil",
    "Energy exports|Naphtha",
    "Energy exports|White spirit & SBP",
    "Energy exports|Lubricants",
    "Energy exports|Bitumen",
    "Energy exports|Paraffin waxes",
    "Energy exports|Petroleum coke",
    "Energy exports|Other oil products",

], append=True
) 

iea_trade.aggregate(
    'Energy imports|Primary Energy|Oil|Volume', 
    [
    "Energy imports|Crude/NGL/feedstocks (if no detail)",
    "Energy imports|Crude oil",
    "Energy imports|Natural gas liquids",
    "Energy imports|Refinery feedstocks",
    "Energy imports|Additives/blending components",
    "Energy imports|Other hydrocarbons",
    "Energy imports|Refinery gas",
    "Energy imports|Ethane",
    "Energy imports|Liquefied petroleum gases (LPG)",
    "Energy imports|Motor gasoline excl. biofuels",
    "Energy imports|Aviation gasoline",
    "Energy imports|Gasoline type jet fuel",
    "Energy imports|Kerosene type jet fuel excl. biofuels",
    "Energy imports|Other kerosene",
    "Energy imports|Gas/diesel oil excl. biofuels",
    "Energy imports|Fuel oil",
    "Energy imports|Naphtha",
    "Energy imports|White spirit & SBP",
    "Energy imports|Lubricants",
    "Energy imports|Bitumen",
    "Energy imports|Paraffin waxes",
    "Energy imports|Petroleum coke",
    "Energy imports|Other oil products",

], append=True
) 

iea_trade.add("Energy exports|Primary Energy|Oil|Volume", "Energy imports|Primary Energy|Oil|Volume", "Trade|Primary Energy|Oil|Volume", append=True)

In [24]:
iea_trade.rename(
    unit={'EJ / yr': 'EJ/yr'},
    inplace=True
)

iea_trade.rename(
    model={'': 'IEA'},
    inplace=True
)

iea_trade.rename(
    scenario={'Historic': 'Historic data'},
    inplace=True
)

In [25]:
iea_trade.to_csv(input_data / "IEA_hist_trade_variables_v2022.csv")

## GDP_NGFS_merge

In [956]:
# TODO: think about converting USD2005 into USD2017!

In [963]:
import pyam
from pathlib import Path
import pandas as pd

data_GDP = pd.read_csv(raw_data / "GDP_NGFS_merged_uncomplete.csv") # this one is SSpD 2023
data_SSP = pd.read_csv(raw_data / "SspDb_country_data_2013-06-12.csv") 

lst_countries = ["SYR", "PSE", "AFG", "VEN"]

In [964]:
missing_data = data_SSP[(data_SSP["REGION"].isin(lst_countries)) & (data_SSP["MODEL"] == "OECD Env-Growth")]
complete = pd.concat([data_GDP, missing_data])
complete.loc[(complete["REGION"].isin(["SYR", "USA"])) & (complete["VARIABLE"] == "GDP|PPP")]
complete["UNIT"] = complete["UNIT"].replace("billion US$2005/yr", "billion USD_2017/yr")
complete["UNIT"].unique()

array(['billion USD_2017/yr', 'million', 'years', 'USD_2017/yr'],
      dtype=object)

In [969]:
complete.loc[(complete["REGION"]=="VEN") & (complete["VARIABLE"]=="GDP|PPP")].columns

Index(['MODEL', 'SCENARIO', 'REGION', 'VARIABLE', 'UNIT', '2025', '2030',
       '2035', '2040', '2045', '2050', '2055', '2060', '2065', '2070', '2075',
       '2080', '2085', '2090', '2095', '2100', '2020', '1950', '1955', '1960',
       '1965', '1970', '1975', '1980', '1985', '1990', '1995', '2000', '2005',
       '2010', '2015', '2105', '2110', '2115', '2120', '2125', '2130', '2135',
       '2140', '2145', '2150'],
      dtype='object')

In [ ]:
# complete.to_csv(input_data / "GDP_NFGS_merge.csv", index=False)
# For now we use what Fabio has sent us

## IEA 2019 CO2 emissions from fuels_ISO.csv: IEA GHG FUEL 2025 -> World_BigCO2

In [50]:
iea_product_mapping = {
    # Aggregates
    "TOTAL": "Total",

    # Coal and coal products
    "HARDCOAL_ND": "Hard coal (if no detail)",
    "BROWNCOAL_ND": "Brown coal (if no detail)",
    "ANTHRACITE": "Anthracite",
    "COKING_COAL": "Coking coal",
    "OTH_BITCOAL": "Other bituminous coal",
    "SUB_BITCOAL": "Sub-bituminous coal",
    "LIGNITE": "Lignite",
    "PATENT_FUEL": "Patent fuel",
    "COKE_OVEN_COKE_OTH": "Coke oven coke",
    "GAS_COKE": "Gas coke",
    "COAL_TAR": "Coal tar",
    "BKB": "BKB",

    # Manufactured & recovered gases
    "GASWORKS_GAS": "Gas works gas",
    "COKE_OVEN_GAS": "Coke oven gas",
    "BLAST_FURNACE_GAS": "Blast furnace gas",
    "OTH_RECOVGASES": "Other recovered gases",

    # Peat and shale
    "PEAT": "Peat",
    "PEAT_PRODUCTS": "Peat products",
    "OIL_SHALE": "Oil shale",

    # Natural gas
    "NATURAL_GAS": "Natural gas",

    # Crude oil, NGL, refinery inputs
    "OIL_PRIM_PRODUCTS_ND": "Crude/NGL/feedstocks (if no detail)",
    "CRUDE_OIL": "Crude oil",
    "NGL": "Natural gas liquids",
    "REFINERY_FEEDSTOCKS": "Refinery feedstocks",

    # Other primary oil inputs
    "ORIMULSION": "Orimulsion",
    "HYDROCARBONS_OTHER": "Other hydrocarbons",

    # Refinery outputs – gases
    "REFINERY_GAS": "Refinery gas",
    "ETHANE": "Ethane",
    "LPG": "Liquefied petroleum gases (LPG)",

    # Transport fuels
    "MOTOR_GASOLINE_NONBIO": "Motor gasoline excl. biofuels",
    "AVIATION_GASOLINE": "Aviation gasoline",
    "GASOLINE_JET": "Gasoline type jet fuel",
    "KEROSENE_JET_NONBIO": "Kerosene type jet fuel excl. biofuels",
    "KEROSENE_OTHER": "Other kerosene",
    "GAS_DIESEL_OIL_NONBIO": "Gas/diesel oil excl. biofuels",

    # Other oil products
    "FUEL_OIL_RESIDUAL": "Fuel oil",
    "NAPHTHA": "Naphtha",
    "WHITE_SPIRIT": "White spirit & SBP",
    "LUBRICANTS": "Lubricants",
    "BITUMEN": "Bitumen",
    "PARAFFIN_WAXES": "Paraffin waxes",
    "PETROLEUM_COKE": "Petroleum coke",
    "OTH_SEC_OIL_PRODS_ND": "Non-specified oil products",

    # Waste (non-renewable)
    "WASTE_INDUSTRIAL_NONREN": "Industrial waste",
    "WASTE_MUNICIPAL_NONREN": "Municipal waste (non-renew)",

    # Memo items – bioenergy
    "PRIMARY_SOLID_BIOFUEL": "Memo: Primary solid biofuels",
    "BIOGASES": "Memo: Biogases",
    "BIOGASOLINE": "Memo: Biogasoline",
    "BIODIESEL": "Memo: Biodiesels",
    "LIQBIOFUEL_OTHER": "Memo: Other liquid biofuels",
    "BIOFUEL_NONSPEC": "Memo: Non-specified primary biofuels & waste",
    "CHARCOAL": "Memo: Charcoal"
}


In [51]:
iea_flow_mapping = {
    # Emissions
    "CO2_FUELCOMB": "CO2 fuel combustion",

    # Electricity & heat production – main activity
    "MAINPROD": "Main activity electricity and heat production",
    "MAINELEC": "Main activity electricity plants",
    "MAINCHP": "Main activity CHP plants",
    "MAINHEAT": "Main activity heat plants",

    # Electricity & heat production – autoproducers
    "AUTOELEC": "Autoproducer electricity plants",
    "AUTOCHP": "Autoproducer CHP plants",
    "AUTOHEAT": "Autoproducer heat plants",
    "UNALLOC_AUTOPROD": "Unallocated autoproducers",

    # Energy industry own use
    "ELECHEAT_GROSS": "Own use in electricity, CHP and heat plants",
    "OTHEN": "Other energy industry own use",

    # Total final consumption
    "TFC": "Memo: Total final consumption",

    # Industry – totals and subsectors
    "TOTIND": "Manufacturing industries and construction",
    "MINING": " Mining and quarrying",
    "CONSTRUC": " Construction",
    "MANUFACT": " Manufacturing",
    "IRONSTL": "  Iron and steel",
    "CHEMICAL": "  Chemical and petrochemical",
    "NONFERR": "  Non-ferrous metals",
    "NONMET": "  Non-metallic minerals",
    "TRANSEQ": "  Transport equipment",
    "MACHINE": "  Machinery",
    "FOODPRO": "  Food and tobacco",
    "PAPERPRO": "  Paper, pulp and printing",
    "WOODPRO": "  Wood and wood products",
    "TEXTILES": "  Textile and leather",
    "INONSPEC": " Industry not elsewhere specified",

    # Transport – totals and modes
    "TOTTRANS": "Transport",
    "ROAD": " Road",
    "DOMESAIR": " Domestic aviation",
    "RAIL": " Rail",
    "PIPELINE": " Pipeline transport",
    "DOMESNAV": " Domestic navigation",
    "TRNONSPE": " Transport not elsewhere specified",

    # Other final consumption sectors
    "RESIDENT": "Residential",
    "COMMPUB": "Commercial and public services",
    "AGRI_FOREST": "Agriculture/forestry",
    "FISHING": "Fishing",
    "ONONSPEC": "Final consumption not elsewhere specified",

    # Memo items – bunkers and power
    "BUNKERS_MARINE": "Memo: International marine bunkers",
    "BUNKERS_AVIATION": "Memo: International aviation bunkers",
    "EPOWERPLT": "Memo: Electricity and heat production"
}


In [43]:
iea_co2_2025 = pd.read_csv(
    raw_data / 'World_BigCO2_2025.csv', # comes from MCG ca_data_management IEA_GHG_FUEL_DETAILED_2025
    encoding="latin1")

iea_co2_2025=iea_co2_2025.drop(0).rename(columns={"Unnamed: 0": "COUNTRY", "Unnamed: 1": "PRODUCT", "TIME":"FLOW"})

In [44]:
iea_co2_2024 = pd.read_csv(
    raw_data / 'World_BigCO2_2024.csv', # comes from MCG ca_data_management IEA_GHG_FUEL_DETAILED_2025
    encoding="latin1")

iea_co2_2024=iea_co2_2024.drop(0).rename(columns={"Unnamed: 0": "COUNTRY", "Unnamed: 1": "PRODUCT", "TIME":"FLOW"})

/var/folders/m8/48479kf16gl9rvhblbkv6myw0000gp/T/ipykernel_66852/2923699552.py:1: DtypeWarning: Columns (44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65) have mixed types. Specify dtype option on import or set low_memory=False.
  iea_co2_2024 = pd.read_csv(


In [52]:
iea_co2 = iea_co2_2025.copy()

In [55]:
iea_co2=iea_co2.replace(iea_flow_mapping)
iea_co2=iea_co2.replace(iea_product_mapping)

In [ ]:
iea_co2["ISO"] = np.nan
iea_co2["COUNTRY"]=iea_co2["COUNTRY"].str.capitalize()

mapping_dict = {c: dt.mapping.getSpatialID(c) for c in iea_co2["COUNTRY"].unique()}
iea_co2["ISO"] = iea_co2["COUNTRY"].map(mapping_dict)

In [63]:
iea_co2=iea_co2.reset_index() # Reset index

# Create a dictionary with `clean` variable names

var_list=iea_co2.PRODUCT.unique() # This is the original variable list
mydict={x:x.strip() for x in var_list} # Create a dictionary with {'original':'clean'} variable name

# Replace variable names in the dictionary
iea_co2['PRODUCT']=iea_co2['PRODUCT'].replace(mydict) # Rename variables in the dataframe `df`

In [64]:
iea_co2=iea_co2.drop("index", axis=1)

In [65]:
iea_co2["REGION"] = np.nan

In [66]:
fuel_dict = pd.read_csv(raw_data / "IEA_Fuel_dict.csv", 
index_col = 0)

In [67]:
mapping = fuel_dict["FUEL"].to_dict()
iea_co2["FUEL"] = iea_co2["PRODUCT"].map(mapping)

In [69]:
iea_co2.to_csv(input_data / "IEA 2019 CO2 emissions from fuels_ISO.csv", index=False) # It is not 2019 data but well 2024 that are being used

## Historical_data: IEA WEB WIND 2025

In [ ]:
# NOTES
# GDP_R_PPP -> not the same unit as what Fabio had, ratio differs from 1.35 for USA, 1.47 for CHN

In [225]:
# Suppose you know the column specs (list of (start, width) tuples) and column names

colspecs = [
    (0, 28),    # COUNTRY
    (28, 59),   # VARIABLE
    (59, 84),   # TIME
    (84, 113),  # UNIT
    (113, 142), # VALUE
    (142, 191), # CODE
    # (191, 209)  # CODE
]

colnames = ['COUNTRY', 'VARIABLE', 'TIME', 'UNIT', 'VALUE', 'CODE']

df = pd.read_fwf(raw_data / "WORLDIND_2025.TXT", colspecs=colspecs, names=colnames) # comes from MCG ca_data_management IEA_WEB_DETAILED_2025
df=df.set_index(["COUNTRY", "VARIABLE", "TIME", "UNIT", "CODE"])

In [226]:
# Define priority order
priority = ['A', 'I', 'P', 'C', 'N', 'O']

# Reset index to make filtering easier
df_reset = df.reset_index()

# Sort by priority
df_reset['priority'] = df_reset['CODE'].apply(lambda x: priority.index(x) if x in priority else len(priority))

# Keep only the highest priority code for each COUNTRY-PRODUCT-FLOW
df_filtered = (
    df_reset.sort_values('priority')
    .groupby(['COUNTRY', 'VARIABLE', 'TIME', 'UNIT'], as_index=False)
    .first()
    .drop(columns='priority')
)

# Set original MultiIndex again if needed
df_filtered = df_filtered.set_index(['CODE', 'COUNTRY', 'VARIABLE', 'TIME', 'UNIT'])

In [227]:
df=df_filtered.copy()

In [228]:
df=df.replace(0, np.nan).dropna(how="all")

In [229]:
df=df.unstack("VARIABLE")
df=df.droplevel(0, axis=1)
df=pd.DataFrame(df.stack())

In [230]:
df_filter=df.loc[isin(UNIT=["MTOE", "TWH", "B_USDPPP", "B_USD", "M_CAP", ])]

In [231]:
df_filter=df_filter.rename(index={
    "POP":"POPULATION", 
    "GDP_R_PPP":"GDP|PPP"})

In [232]:
df_filter=df_filter.loc[isin(VARIABLE = ["GDP|PPP", "POPULATION", "TFC"])]
df_filter=df_filter.droplevel(["UNIT", "CODE"]).unstack("VARIABLE")
df_filter=df_filter.droplevel(0, axis=1)
df_filter.columns.name = ""
df_filter=df_filter.reset_index()

In [233]:
df_filter["COUNTRY"] = df_filter["COUNTRY"].str.lower().str.capitalize()

In [234]:
mapping_dict = {c: get_iso_code(c) for c in df_filter["COUNTRY"].unique()}
df_filter["ISO"] = df_filter["COUNTRY"].map(mapping_dict)

In [235]:
iso_code = list(set([x for x in df_filter.ISO if len(x)==3]))
df_filter=df_filter.loc[df_filter["ISO"].isin(iso_code)]

In [238]:
df_filter[df_filter.TIME==1971]

,COUNTRY,TIME,GDP|PPP,TFC,POPULATION,ISO
54,Albania,1971,9.577,1.416,2.188,ALB
108,Algeria,1971,106.700,2.126,14.099,DZA
216,Angola,1971,35.624,2.899,5.991,AGO
270,Argentina,1971,473.000,22.826,24.257,ARG
430,Australia,1971,329.446,36.076,13.198,AUS
...,...,...,...,...,...,...
9242,Venezuela,1971,248.700,9.507,11.741,VEN
9296,Vietnam,1971,82.207,12.378,42.449,VNM
9510,Yemen,1971,9.991,0.322,7.257,YEM
9564,Zambia,1971,15.099,3.009,4.443,ZMB


In [ ]:
df_filter.to_csv(
    input_data / "Historical_data.csv", 
    index=False)

In [774]:
df_filter

,COUNTRY,TIME,GDP|PPP,TFC,POPULATION,ISO
54,Albania,1971,9.577,1.416,2.188,ALB
55,Albania,1972,10.431,1.606,2.243,ALB
56,Albania,1973,11.575,1.420,2.297,ALB
57,Albania,1974,11.336,1.508,2.350,ALB
58,Albania,1975,12.002,1.660,2.405,ALB
...,...,...,...,...,...,...
9667,Zimbabwe,2020,54.530,5.314,15.527,ZWE
9668,Zimbabwe,2021,59.148,5.558,15.797,ZWE
9669,Zimbabwe,2022,62.779,6.209,16.069,ZWE
9670,Zimbabwe,2023,66.129,6.379,16.341,ZWE


## PLATTS

In [ ]:
platts = pd.read_excel(raw_data / "ALLUNITS_PLATTS.xlsx")

/Users/marie-charlottegeffray/opt/anaconda3/envs/1o5/lib/python3.9/site-packages/openpyxl/worksheet/_read_only.py:79: UserWarning: Unknown extension is not supported and will be removed
  for idx, row in parser.parse():


In [439]:
mapping_dict = {c: get_iso_code(c) for c in platts["COUNTRY"].unique()}
platts["ISO"] = platts["COUNTRY"].map(mapping_dict)

In [ ]:
# platts.to_excel(
#     input_data / "ALLUNITS_PLATTS_ISO.xlsx", 
#     index=False)

In [ ]:
platts.to_csv(
    input_data / "ALLUNITS_PLATTS_ISO.csv", 
) # -> YOU NEED THE CSV

# PROJECTIONS

## REMIND Data

In [158]:
remind = pd.read_excel(raw_data / "REMIND-MAgPIE_coreAndVariations_2025-12-10_16.07.26.xlsx", index_col = [0,1,2,3,4])

In [159]:
remind.index.names = ['MODEL', 'SCENARIO', 'REGION', 'VARIABLE', 'UNIT']

In [160]:
remind[[2021,2022,2023,2024]]=np.nan

In [161]:
remind.columns=remind.columns.astype(int)

In [163]:
remind=remind.sort_index(axis=1).interpolate(axis=1)

In [166]:
remind.reset_index().to_csv(project_data / "snapshot_v1/REMIND.csv")

## Update Scenario config file

In [ ]:
scenario_config = pd.read_csv(project_data / "project_folder/scenario_config.csv")

In [ ]:
scenario_config.drop(4).to_csv(project_data / "scenario_config.csv")